# EEG Trajectory-Based BabelBrain Simulations Using CT- and ZTE-Derived Skull Masks

**Adapted from:** Zadeh et al. (2025), DOI: 10.1088/1741-2552/adab22

This notebook runs EEG trajectory-based BabelBrain simulations. Comments were added to clarify the workflow only.

**Main workflow:**  
Load EEG normals → convert to Brainsight-format targets → generate masks → run CT acoustic simulations → run ZTE acoustic simulations → collect and summarize outputs.

---

## Ultrasound Simulations Using the CT-Based and Zero TE-Based Skull Masks

The section below is adapted and copied from Zadeh et al. (2025).

### Context

This notebook shows an example workflow that uses targets derived from EEG locations calculated from SimNIBS CHARM outputs. Enhanced SimNIBS tissue segmentation is used to improve tissue classification, including fat segmentation.

In this demonstration, BabelBrain v0.8.0 domain generation and transcranial ultrasound modeling are executed for EEG-derived targets.

This script can be modified for any target of interest, depending on the research aim.

### Dataset

The dataset includes participants with the following skull density ratio (SDR) values:

0.615, 0.598, 0.537, 0.527, 0.521, 0.517, 0.515, 0.514, 0.479, 0.470, 0.455, 0.416, 0.410, and 0.385.

### Create List of EEG Locations

Run `1_eeg_normals.py` in a conda environment based on SimNIBS, specifying the paths to the SimNIBS output files for each participant.

Example:

```bash
conda activate <path_to_simnibs>/simnibs_env/

python 0_eeg_normals.py \
~/Documents/TempForSim/SDR_0p42/m2m_SDR_0p42/SDR_0p42.msh \
~/Documents/TempForSim/SDR_0p42/m2m_SDR_0p42/eeg_positions/EEG10-10_Neuroelectrics.csv \
SDR_0p42_eeg_normals.csv

In [ ]:
import numpy as np
import pandas as pd
import sys
import os
import time
from multiprocessing import Process, Queue
import multiprocessing
import nibabel
from pathlib import Path
from glob import glob
from scipy import ndimage
import shutil
from pathlib import Path

## Additions to path
Add the paths to BabelBrain

In [ ]:
sys.path.append('/Users/moonjeong/Documents/GitHub/BabelBrain')     #Define the path to the BabelBrain folder
sys.path.append('/Users/moonjeong/Documents/GitHub/BabelBrain/BabelBrain')

In [ ]:
from BabelBrain.CalculateMaskProcess import CalculateMaskProcess
from TranscranialModeling.BabelIntegrationBASE import GetSmallestSOS
from BabelBrain.CalculateFieldProcess import CalculateFieldProcess
from BabelBrain.Babel_Thermal.CalculateThermalProcess import CalculateThermalProcess
from ThermalModeling.CalculateTemperatureEffects import GetThermalOutName

## Supportive functions
To read EEG poistions and its normals, and create Brainsight compatible trajectory definitions

In [ ]:
def rotation_matrix_from_vectors(vec1, vec2):
    """ Find the rotation matrix that aligns vec1 to vec2
    :param vec1: A 3d "source" vector
    :param vec2: A 3d "destination" vector
    :return mat: A transform matrix (3x3) which when applied to vec1, aligns it with vec2.
    """
    a, b = (vec1 / np.linalg.norm(vec1)).reshape(3), (vec2 / np.linalg.norm(vec2)).reshape(3)
    v = np.cross(a, b)
    if any(v): #if not all zeros then 
        c = np.dot(a, b)
        s = np.linalg.norm(v)
        kmat = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
        return np.eye(3) + kmat + kmat.dot(kmat) * ((1 - c) / (s ** 2))

    else:
        return np.eye(3) #cross of all zeros only occurs on identical directions

In [ ]:
templateBSight=\
'''# Version: 13
# Coordinate system: NIfTI:Aligned
# Created by: Brainsight 2.5
# Units: millimetres, degrees, milliseconds, and microvolts
# Encoding: UTF-8
# Notes: Each column is delimited by a tab. Each value within a column is delimited by a semicolon.
# Target Name	Loc. X	Loc. Y	Loc. Z	m0n0	m0n1	m0n2	m1n0	m1n1	m1n2	m2n0	m2n1	m2n2
'''
lineBSight=\
'''{name}\t{X:10.9f}\t{Y:10.9f}\t{Z:10.9f}\t{m0n0:10.9f}\t{m0n1:10.9f}\t{m0n2:10.9f}\t{m1n0:10.9f}\t{m1n1:10.9f}\t{m1n2:10.9f}\t{m2n0:10.9f}\t{m2n1:10.9f}\t{m2n2:10.9f}
'''

**Locations**
----
Specific locations (Fp1, Fp2, Fpz, LPA, Nz, P9, P10, RPA, Afz, AF8, AF7, AF4, AF3, Iz) were omitted due to complications such as passage through air-filled structures, proximity to the ear, or mask generation issues attributed to MRI scan defacing. Consequently, 53 EEG locations were selected for analysis.

In [ ]:
## this file contains YES and NO for each EEG landmark to indicate if we want to include it
SELECTED_FOR_FUS = pd.read_csv('/Volumes/Trans_2/transmission/scripts/EnhanceTUS_Zadeh_JNE-main/SELECTED_EEG.csv',index_col='Name')['Include'].to_dict()

In [ ]:
def ConvertEEGToBSightFUS(inputcsv='ernie_eeg_normals.csv',
                          outputBSight='ernie_bsight.txt',
                          depthFromSkin=0.0):
    MatResults={}
    A=pd.read_csv(inputcsv)
    with open(outputBSight,'w') as f:
        f.write(templateBSight)
        for index,entry in A.iterrows():
            transform=np.eye(4)
            NormalV=np.array(entry[['Nx','Ny','Nz']].tolist())
            transform[:3,:3]=rotation_matrix_from_vectors(np.array([0,0,-1]),NormalV)
            transform[:3,3]=np.array(entry[['R','A','S']].tolist())+depthFromSkin*NormalV
            outString=lineBSight.format(m0n0=transform[0,0],
                                    m0n1=transform[1,0],
                                    m0n2=transform[2,0],
                                    m1n0=transform[0,1],
                                    m1n1=transform[1,1],
                                    m1n2=transform[2,1],
                                    m2n0=transform[0,2],
                                    m2n1=transform[1,2],
                                    m2n2=transform[2,2],
                                    X=transform[0,3],
                                    Y=transform[1,3],
                                    Z=transform[2,3],
                                    name=entry['Name'])
            if SELECTED_FOR_FUS[entry['Name']]=='YES':
                f.write(outString)
                MatResults[entry['Name']]=transform
    return MatResults

def SaveMatFile(outfile,ID,transform):
    with open(outfile,'w') as f:
        f.write(templateBSight)
        outString=lineBSight.format(m0n0=transform[0,0],
                                m0n1=transform[1,0],
                                m0n2=transform[2,0],
                                m1n0=transform[0,1],
                                m1n1=transform[1,1],
                                m1n2=transform[2,1],
                                m2n0=transform[0,2],
                                m2n1=transform[1,2],
                                m2n2=transform[2,2],
                                X=transform[0,3],
                                Y=transform[1,3],
                                Z=transform[2,3],
                                name=ID)
        f.write(outString)

## Function to generate input files from trajectories
It calls the same code as in the BabelBrain's GUI step 1

In [ ]:
def RunMaskGeneration(T1W,
                      ID,
                      simbnibs_path,
                      Mat4Trajectory,
                      ComputingDevice='M2',
                      ComputingBackend=3,
                      Frequency=750e3,
                      BasePPW=6,
                      TxSystem='Single',
                      bUseCT=True,
                      CT_or_ZTE_input='',
                      ZTERange=[0.1,0.65],
                      HUThreshold=300.0,
                      bReuseFiles=True,
                      CTType=1):

    print("*"*40)
    print("*"*5+" Calculating mask.. BE PATIENT... it can take a couple of minutes...")
    print("*"*40)

    deviceName = ComputingDevice
    COMPUTING_BACKEND = ComputingBackend

    T1WIso = T1W
    SmallestSoS = GetSmallestSOS(Frequency, bShear=True)

    # Base prefix
    prefix = ID + '_' + TxSystem + '_%ikHz_%iPPW_' % (int(Frequency/1e3), BasePPW)

    # Tag prefix by modality (our tag)
    mod_tag = "ZTE" if (bUseCT and int(CTType) in [2, 3]) else "CT"
    prefix = prefix + f"{mod_tag}_"

    SpatialStep = np.round(SmallestSoS / Frequency / BasePPW * 1e3, 3)  # mm
    print("Frequency, SmallestSoS, BasePPW, SpatialStep", Frequency, SmallestSoS, BasePPW, SpatialStep)
    print("Mat4Trajectory", Mat4Trajectory)

    kargs = {}
    kargs['SimbNIBSDir'] = simbnibs_path
    kargs['SimbNIBSType'] = 'charm'
    kargs['CoregCT_MRI'] = True
    kargs['TrajectoryType'] = 'brainsight'
    kargs['Mat4Trajectory'] = Mat4Trajectory
    kargs['T1Source_nii'] = T1W
    kargs['T1Conformal_nii'] = T1WIso
    kargs['nIterationsAlign'] = 10
    kargs['SpatialStep'] = SpatialStep
    kargs['InitialAligment'] = 'HF'
    kargs['Location'] = [0, 0, 0]
    kargs['prefix'] = prefix
    kargs['bPlot'] = False
    kargs['bAlignToSkin'] = True

    if bUseCT:
        kargs['CT_or_ZTE_input'] = CT_or_ZTE_input
        kargs['CTType'] = CTType
        if int(kargs['CTType']) in [2, 3]:
            kargs['ZTERange'] = ZTERange
        kargs['HUThreshold'] = HUThreshold

    queue = Queue()
    print(COMPUTING_BACKEND, deviceName, kargs)

    maskWorkerProcess = Process(
        target=CalculateMaskProcess,
        args=(queue, COMPUTING_BACKEND, deviceName),
        kwargs=kargs
    )
    maskWorkerProcess.start()

    T0 = time.time()
    bNoError = True
    while maskWorkerProcess.is_alive():
        time.sleep(0.1)
        while not queue.empty():
            cMsg = queue.get()
            print(cMsg, end='')
            if '--Babel-Brain-Low-Error' in cMsg:
                bNoError = False

    maskWorkerProcess.join()
    while not queue.empty():
        cMsg = queue.get()
        print(cMsg, end='')
        if '--Babel-Brain-Low-Error' in cMsg:
            bNoError = False

    if not bNoError:
        print("*"*40)
        print("*"*5+" Error in execution.")
        print("*"*40)
        raise RuntimeError("Error when generating mask!!")

    TEnd = time.time()
    print('Total time', TEnd - T0)
    print("*"*40)
    print("*"*5+" DONE calculating mask.")
    print("*"*40)

    # Rename the files: _CT_CT -> _CT (BabelBrain appends CT internally so naming CT and ZTE in your code will name the file two times with _CT_CT ot _ZTE_CT)
    if bUseCT and int(CTType) == 1:
        out_dir = Path(Mat4Trajectory).parent  # same folder as your brainsight txt
        hits = list(out_dir.glob("*_CT_CT*"))
        print(f"[rename] found {len(hits)} files with _CT_CT in {out_dir}")
    
        for p in hits:
            new_path = p.with_name(p.name.replace("_CT_CT", "_CT"))
            if new_path.exists():
                print(f"[rename] SKIP exists: {new_path.name}")
                continue
            p.rename(new_path)
            print(f"[rename] {p.name} -> {new_path.name}")


    # FORCE FIX: rename *_ZTE_CT* -> *_ZTE* (BabelBrain appends CT internally)
    if bUseCT and int(CTType) in [2, 3]:
        out_dir = Path(Mat4Trajectory).parent  # same folder as your brainsight txt
        hits = list(out_dir.glob("*_ZTE_CT*"))
        print(f"[rename] found {len(hits)} files with _ZTE_CT in {out_dir}")
    
        for p in hits:
            new_path = p.with_name(p.name.replace("_ZTE_CT", "_ZTE"))
            if new_path.exists():
                print(f"[rename] SKIP exists: {new_path.name}")
                continue
            p.rename(new_path)
            print(f"[rename] {p.name} -> {new_path.name}")

## Function to run transcranial ultrasound modelling
It calls the same code as in the BabelBrain's GUI step 2

In [ ]:
def DistanceOutPlaneToFocus(FocalLength,Diameter):
    return np.sqrt(FocalLength**2-(Diameter/2)**2)

def RunAcousticSim(
    T1W,
    ID,
    TxSystem='Single',
    deviceName='M2',
    Aperture=50e-3, #in m
    FocalLength=50e-3, #in m
    Frequency=750e3, #in Hz
    PPW=6,
    COMPUTING_BACKEND=3,
    zLengthBeyonFocalPointWhenNarrow=40e-3, #distance to run simulation beyond target
    TxAdjustmentZ=0.0,
    bUseCT=True,
    ModalityTag="CT"
):
    basedir, pID = os.path.split(os.path.split(T1W)[0])
    Target = [ID + '_' + TxSystem]

    InputSim = os.path.join(
        os.path.split(T1W)[0],
        f"{ID}_{TxSystem}_{int(Frequency/1e3)}kHz_{PPW}PPW_{ModalityTag}_BabelViscoInput.nii.gz"
    )

    print('InputSim', InputSim, basedir, pID)

    MaskData = nibabel.load(InputSim)
    FinalMask = np.flip(MaskData.get_fdata(), axis=2)
    VoxelSize = MaskData.header.get_zooms()[0] / 1e3

    TargetLocation = np.array(np.where(FinalMask == 5.0)).flatten()
    LineOfSight = FinalMask[TargetLocation[0], TargetLocation[1], :]
    StartSkin = np.where(LineOfSight > 0)[0].min()

    DistanceFromSkin = (TargetLocation[2] - StartSkin) * VoxelSize

    Diameter = Aperture
    DOut = DistanceOutPlaneToFocus(FocalLength, Diameter)   # meters
    ZMax = DOut - DistanceFromSkin                          # meters
    ZMaxSkin = np.round(ZMax * 1e3, 1) / 1e3               # meters, rounded to 0.1 mm

    TxMechanicalAdjustmentX = 0.0
    TxMechanicalAdjustmentY = 0.0
    TxMechanicalAdjustmentZ = ZMaxSkin + TxAdjustmentZ

    ZIntoSkin = 0.0
    CurDistance = ZMaxSkin - TxMechanicalAdjustmentZ
    if CurDistance < 0:
        ZIntoSkin = np.abs(CurDistance)

    extrasuffix = 'Foc%03.1f_Diam%03.1f_' % (FocalLength * 1e3, Aperture * 1e3)

    Frequencies = [Frequency]
    basePPW = [PPW]

    kargs = {}
    kargs['extrasuffix'] = extrasuffix
    kargs['ID'] = pID
    kargs['deviceName'] = deviceName
    kargs['COMPUTING_BACKEND'] = COMPUTING_BACKEND
    kargs['basePPW'] = basePPW
    kargs['basedir'] = basedir + os.sep
    kargs['Aperture'] = Aperture
    kargs['FocalLength'] = FocalLength
    kargs['TxMechanicalAdjustmentZ'] = TxMechanicalAdjustmentZ
    kargs['TxMechanicalAdjustmentX'] = TxMechanicalAdjustmentX
    kargs['TxMechanicalAdjustmentY'] = TxMechanicalAdjustmentY
    kargs['ZIntoSkin'] = ZIntoSkin
    kargs['Frequencies'] = Frequencies
    kargs['zLengthBeyonFocalPointWhenNarrow'] = zLengthBeyonFocalPointWhenNarrow
    kargs['bUseCT'] = bUseCT
    kargs['bPETRA'] = False
    kargs['bMinimalSaving'] = False
    kargs['bUseRayleighForWater'] = True

    bNoError = True
    queue = Queue()
    T0 = time.time()

    fieldWorkerProcess = Process(
        target=CalculateFieldProcess,
        args=(queue, Target, TxSystem),
        kwargs=kargs
    )

    fieldWorkerProcess.start()

    while fieldWorkerProcess.is_alive():
        time.sleep(0.1)
        while not queue.empty():
            cMsg = queue.get()
            print(cMsg, end='')
            if '--Babel-Brain-Low-Error' in cMsg:
                bNoError = False

    fieldWorkerProcess.join()

    while not queue.empty():
        cMsg = queue.get()
        print(cMsg, end='')
        if '--Babel-Brain-Low-Error' in cMsg:
            bNoError = False

    if bNoError:
        print('Total time', time.time() - T0)
        print("*" * 40)
        print("*" * 5 + " DONE ultrasound simulation.")
        print("*" * 40)
    else:
        print("*" * 40)
        print("*" * 5 + " Error in execution.")
        print("*" * 40)
        raise RuntimeError("Error when running transcranial simulation!!")

### Convert all  EEG locations 

In [ ]:
eeg_files = [
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9079_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9085_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9088_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9089_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9090_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9091_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9093_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9094_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9095_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9096_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9097_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9098_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9099_eeg_normals.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/m2m_9100_eeg_normals.csv'
]

output_files = [
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9079_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9085_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9088_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9089_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9090_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9091_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9093_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9094_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9095_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9096_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9097_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9098_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9099_eeg_normals_35mm.csv',
    '/Volumes/Trans_2/transmission/data/eeg_normals/sub-9100_eeg_normals_35mm.csv'
]

In [ ]:
TxSystem = 'Single'

frequency  = 750e3          
frequencies = [frequency]   

PPW   = 6
PPWs  = {frequency: PPW} 

Aperture = 50e-3
FocalLength = 50e-3
zLengthBeyonFocalPointWhenNarrow = 80e-3
depthFromSkin = 30.0
ComputingDevice = 'M2'

root_path = '/Volumes/Trans_2'

In [ ]:
simbnibs_paths = [
   root_path + '/transmission/data/SimNIBS/m2m_9079',
   root_path + '/transmission/data/SimNIBS/m2m_9085',
   root_path + '/transmission/data/SimNIBS/m2m_9088',
   root_path + '/transmission/data/SimNIBS/m2m_9089',
   root_path + '/transmission/data/SimNIBS/m2m_9090',
   root_path + '/transmission/data/SimNIBS/m2m_9091',
   root_path + '/transmission/data/SimNIBS/m2m_9093',
   root_path + '/transmission/data/SimNIBS/m2m_9094',
   root_path + '/transmission/data/SimNIBS/m2m_9095',
   root_path + '/transmission/data/SimNIBS/m2m_9096',
   root_path + '/transmission/data/SimNIBS/m2m_9097',
   root_path + '/transmission/data/SimNIBS/m2m_9098',
   root_path + '/transmission/data/SimNIBS/m2m_9099',
   root_path + '/transmission/data/SimNIBS/m2m_9100'
    
]

In [ ]:
t1w_files = [
   root_path + '/transmission/data/CT/sub-9079/anat/sub-9079_ses-01_T1w.nii.gz',
   root_path + '/transmission/data/CT/sub-9085/anat/sub-9085_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/CT/sub-9088/anat/sub-9088_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/CT/sub-9089/anat/sub-9089_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/CT/sub-9090/anat/sub-9090_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/CT/sub-9091/anat/sub-9091_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/CT/sub-9093/anat/sub-9093_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/CT/sub-9094/anat/sub-9094_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/CT/sub-9095/anat/sub-9095_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/CT/sub-9096/anat/sub-9096_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/CT/sub-9097/anat/sub-9097_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/CT/sub-9098/anat/sub-9098_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/CT/sub-9099/anat/sub-9099_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/CT/sub-9100/anat/sub-9100_ses-01_T1w.nii.gz'
    
]

CT_files = [
    root_path + '/transmission/data/CT/sub-9079/anat/sub-9079_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9085/anat/sub-9085_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9088/anat/sub-9088_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9089/anat/sub-9089_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9090/anat/sub-9090_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9091/anat/sub-9091_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9093/anat/sub-9093_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9094/anat/sub-9094_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9095/anat/sub-9095_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9096/anat/sub-9096_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9097/anat/sub-9097_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9098/anat/sub-9098_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9099/anat/sub-9099_ses-01_CT.nii.gz',
    root_path + '/transmission/data/CT/sub-9100/anat/sub-9100_ses-01_CT.nii.gz'
    
]

## CT-based Acoustic Simulation

In [ ]:
import nibabel as nib

def qa_check_brainsight_txt(txt_path, expected_id):
    txt_path = Path(txt_path)

    if not txt_path.exists():
        raise FileNotFoundError(f"Missing Brainsight txt: {txt_path}")

    with open(txt_path, "r") as f:
        lines = f.readlines()

    nonempty = [x.strip() for x in lines if x.strip()]
    hits = [line for line in nonempty if str(expected_id) in line]

    print("\n[QA][Brainsight]")
    print(f"  file: {txt_path.name}")
    print(f"  expected ID: {expected_id}")
    print(f"  lines containing ID: {len(hits)}")

    for line in hits[:5]:
        print(f"    {line}")

    if len(hits) == 0:
        raise ValueError(f"{expected_id} not found in {txt_path.name}")

def qa_check_mask(mask_path, expected_id):
    mask_path = Path(mask_path)

    if not mask_path.exists():
        raise FileNotFoundError(f"Missing mask file: {mask_path}")

    img = nib.load(str(mask_path))
    data = img.get_fdata()

    target_vox = np.argwhere(data == 5.0)

    print("\n[QA][Mask]")
    print(f"  ID: {expected_id}")
    print(f"  mask: {mask_path.name}")
    print(f"  shape: {data.shape}")
    print(f"  target voxel count (label 5): {len(target_vox)}")

    if len(target_vox) == 0:
        raise ValueError(f"No label-5 target voxels found in {mask_path.name}")

    centroid_vox = target_vox.mean(axis=0)
    affine = img.affine
    centroid_mm = nib.affines.apply_affine(affine, centroid_vox)

    print(f"  centroid voxel: {np.round(centroid_vox, 2)}")
    print(f"  centroid world/mm: {np.round(centroid_mm, 2)}")

    return {
        "id": str(expected_id),
        "mask": str(mask_path),
        "n_target_vox": int(len(target_vox)),
        "centroid_vox": centroid_vox,
        "centroid_mm": centroid_mm
    }

# Main Loop
qa_results = []

for freq in frequencies:
    PPW = PPWs[freq]
    kHz = int(round(freq / 1e3))

    for T1W, eeg_csv, ct_file, sim_path in zip(t1w_files, eeg_files, CT_files, simbnibs_paths):
        T1W = Path(T1W)
        anat_dir = T1W.parent

        print("\n" + "="*60)
        print(f"[SUBJECT] T1W = {T1W.name}")
        print(f"[SUBJECT] EEG = {eeg_csv}")
        print(f"[SUBJECT] CT  = {ct_file}")
        print(f"[SUBJECT] SIMBNIBS = {sim_path}")
        print("="*60)

        # Build the all-target file once, only to recover IDs
        MatResults = ConvertEEGToBSightFUS(
            inputcsv=eeg_csv,
            outputBSight=str(anat_dir / "brainsight.txt"),
            depthFromSkin=depthFromSkin
        )

        # Read EEG CSV
        try:
            df = pd.read_csv(eeg_csv)
        except Exception:
            df = pd.read_csv(eeg_csv, sep=None, engine="python")

        # Detect target ID column
        id_col = None
        for c in [
            "Target Name", "target_name", "Target", "target",
            "Name", "name", "ID", "Id", "id",
            "Electrode", "electrode", "Label", "label"
        ]:
            if c in df.columns:
                id_col = c
                break

        if id_col is None:
            raise ValueError(
                f"Could not detect target ID column in EEG CSV: {eeg_csv}\n"
                f"Columns = {list(df.columns)}"
            )

        print(f"[INFO] Using ID column: {id_col}")
        print(f"[INFO] IDs from MatResults: {list(MatResults.keys())}")

        # Loop through each target separately
        for ID in MatResults:
            print("\n" + "-"*50)
            print(f"[TARGET] ID = {ID}")
            print("-"*50)

            # Isolate one target row
            row_df = df[df[id_col].astype(str) == str(ID)]

            if len(row_df) == 0:
                raise ValueError(f"Could not find ID={ID} in EEG CSV: {eeg_csv}")
            if len(row_df) > 1:
                raise ValueError(f"Multiple rows found for ID={ID} in EEG CSV: {eeg_csv}")

            # Save one-target csv
            single_csv = anat_dir / f"brainsight_{ID}.csv"
            row_df.to_csv(single_csv, index=False)
            print(f"[WRITE] single CSV: {single_csv.name}")

            # Convert one-target csv -> one-target brainsight txt
            single_bsight = anat_dir / f"brainsight_{ID}.txt"
            ConvertEEGToBSightFUS(
                inputcsv=str(single_csv),
                outputBSight=str(single_bsight),
                depthFromSkin=depthFromSkin
            )
            print(f"[WRITE] single Brainsight txt: {single_bsight.name}")

            qa_check_brainsight_txt(single_bsight, ID)

            # run mask generation
            RunMaskGeneration(
                T1W=str(T1W),
                ID=ID,
                simbnibs_path=str(sim_path),
                Mat4Trajectory=str(single_bsight), 
                ComputingDevice=ComputingDevice,
                ComputingBackend=3,
                Frequency=freq,
                BasePPW=PPW,
                TxSystem=TxSystem,
                bUseCT=True,
                CT_or_ZTE_input=str(ct_file),
                ZTERange=[0.1, 0.65],
                HUThreshold=300.0,
                bReuseFiles=False,                
                CTType=1
            )

            base = f"{ID}_{TxSystem}_{kHz}kHz_{PPW}PPW"
            mask_path = anat_dir / f"{base}_CT_BabelViscoInput.nii.gz"

            qa_result = qa_check_mask(mask_path, ID)
            qa_results.append(qa_result)

print("\n" + "="*70)
print("FINAL QA SUMMARY")
print("="*70)

for r in qa_results:
    print(
        f"ID={r['id']:<8} "
        f"n_target_vox={r['n_target_vox']:<8} "
        f"centroid_vox={np.round(r['centroid_vox'], 2)} "
        f"centroid_mm={np.round(r['centroid_mm'], 2)}"
    )

seen = {}
for r in qa_results:
    key = tuple(np.round(r["centroid_vox"], 2))
    seen.setdefault(key, []).append(r["id"])

print("\nPotential duplicate targets:")
found_duplicate = False
for key, ids in seen.items():
    if len(ids) > 1:
        found_duplicate = True
        print(f"  SAME centroid {key} -> IDs {ids}")

if not found_duplicate:
    print("  No duplicate centroids detected.")

print("\nCT mask generation + QA done.")

## Prepare CT Acoustic Simulation Inputs

This block scans the CT-tagged BabelBrain input files and creates the generic filenames expected by `RunAcousticSim`. It checks that the required CT mask, air-region mask, CT image, and calibration files exist before copying only the missing generic input files.

In [ ]:
from pathlib import Path
import shutil
import nibabel as nib
import numpy as np

for freq in frequencies:
    PPW = PPWs[freq]
    kHz = int(freq / 1e3)

    for T1W, eeg_csv in zip(t1w_files, eeg_files):
        T1W = Path(T1W)
        anat_dir = T1W.parent

        MatResults = ConvertEEGToBSightFUS(
            inputcsv=eeg_csv,
            outputBSight=str(anat_dir / "brainsight.txt"),
            depthFromSkin=depthFromSkin
        )

        for ID in MatResults:
            base = f"{ID}_{TxSystem}_{kHz}kHz_{PPW}PPW"

            # Scan CT-tagged files
            src_mask = anat_dir / f"{base}_CT_BabelViscoInput.nii.gz"
            src_air  = anat_dir / f"{base}_CT_AirRegions.nii.gz"
            src_ct   = anat_dir / f"{base}_CT.nii.gz"
            src_cal  = anat_dir / f"{base}_CT-cal.npz"

            for p in (src_mask, src_air, src_ct, src_cal):
                if not p.exists():
                    raise FileNotFoundError(f"Missing CT file: {p}")

            # Create only the generic names the acoustic code tries to load
            dst_mask = anat_dir / f"{base}_BabelViscoInput.nii.gz"
            dst_air  = anat_dir / f"{base}_AirRegions.nii.gz"
            dst_ct   = anat_dir / f"{base}_CT.nii.gz"
            dst_cal  = anat_dir / f"{base}_CT-cal.npz"

            if not dst_mask.exists():
                shutil.copyfile(src_mask, dst_mask)
            if not dst_air.exists():
                shutil.copyfile(src_air, dst_air)
                
print("CT scan done (generic inputs created only where missing).")

## Run CT-based Acoustic Simulations

This block runs the CT-based acoustic simulations for each EEG-derived target and frequency. It converts each EEG trajectory into a Brainsight-style target, checks the CT acoustic input mask, verifies the target voxel label, and then runs the CT acoustic simulation workflow.

In [ ]:
from pathlib import Path
import nibabel as nib
import numpy as np

for freq in frequencies:
    PPW = PPWs[freq]
    kHz = int(round(freq / 1e3))
    foc_mm = float(FocalLength) * 1e3
    diam_mm = float(Aperture) * 1e3

    for T1W, eeg_csv in zip(t1w_files, eeg_files):
        T1W = Path(T1W)
        anat_dir = T1W.parent

        MatResults = ConvertEEGToBSightFUS(
            inputcsv=eeg_csv,
            outputBSight=str(T1W.parent / "brainsight.txt"),
            depthFromSkin=depthFromSkin
        )

        for ID in MatResults:
            base = f"{ID}_{TxSystem}_{kHz}kHz_{PPW}PPW"

            # Acoustic code for CT actually reads the CT-tagged mask
            ct_mask = anat_dir / f"{base}_CT_BabelViscoInput.nii.gz"
            if not ct_mask.exists():
                raise FileNotFoundError(f"Missing acoustic input mask: {ct_mask}")

            # Inspect the actual mask that RunAcousticSim will use
            img = nib.load(str(ct_mask))
            data = np.flip(img.get_fdata(), axis=2)

            pts = np.argwhere(data == 5.0)

            print("\n" + "=" * 90)
            print(f"[DEBUG] Subject : {T1W.name}")
            print(f"[DEBUG] ID      : {ID}")
            print(f"[DEBUG] Freq    : {freq}")
            print(f"[DEBUG] PPW     : {PPW}")
            print(f"[DEBUG] Mask    : {ct_mask.name}")
            print(f"[DEBUG] #voxels labeled 5 : {pts.shape[0]}")

            if pts.shape[0] == 0:
                raise RuntimeError(f"No target voxel (label 5) found in {ct_mask}")

            print(f"[DEBUG] first target voxel(s): {pts[:10]}")

            if pts.shape[0] > 1:
                print(f"[WARNING] Multiple target voxels found for {ID}. Acoustic code may be using the first one only.")

            target_preview = pts[0]
            line_of_sight = data[target_preview[0], target_preview[1], :]
            skin_idx = np.where(line_of_sight > 0)[0]

            if len(skin_idx) == 0:
                print(f"[WARNING] No skin found along line of sight for {ID}")
                start_skin = None
                distance_from_skin_mm = None
            else:
                start_skin = skin_idx.min()
                voxel_sizes = img.header.get_zooms()[:3]
                voxel_size_mm = float(voxel_sizes[0])
                distance_from_skin_mm = (target_preview[2] - start_skin) * voxel_size_mm

            print(f"[DEBUG] preview TargetLocation : {target_preview}")
            print(f"[DEBUG] preview StartSkin      : {start_skin}")
            print(f"[DEBUG] preview DistanceFromSkin (mm): {distance_from_skin_mm}")
            print("=" * 90)

            # remove stale acoustic outputs before rerun
            old_patterns = [
                f"{base}_Foc{foc_mm:.1f}_Diam{diam_mm:.1f}_DataForSim*.h5",
                f"{base}_Foc{foc_mm:.1f}_Diam{diam_mm:.1f}_Water_DataForSim*.h5",
            ]
            for pat in old_patterns:
                for oldf in anat_dir.glob(pat):
                    try:
                        oldf.unlink()
                        print(f"[deleted old acoustic output] {oldf.name}")
                    except Exception as e:
                        print(f"[warning] could not delete {oldf.name}: {e}")

            RunAcousticSim(
                T1W=str(T1W),
                ID=ID,
                TxSystem=TxSystem,
                deviceName=ComputingDevice,
                Aperture=Aperture,
                FocalLength=FocalLength,
                Frequency=freq,
                PPW=PPW,
                COMPUTING_BACKEND=3,
                zLengthBeyonFocalPointWhenNarrow=zLengthBeyonFocalPointWhenNarrow,
                bUseCT=True
            )

print("CT acoustic done.")

## ZTE-based Acoustic Simulation

In [ ]:
t1w_files = [
    root_path + '/transmission/data/ZTE/sub-9079/anat/sub-9079_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9085/anat/sub-9085_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9088/anat/sub-9088_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9089/anat/sub-9089_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9090/anat/sub-9090_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9091/anat/sub-9091_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9093/anat/sub-9093_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9094/anat/sub-9094_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9095/anat/sub-9095_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9096/anat/sub-9096_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9097/anat/sub-9097_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9098/anat/sub-9098_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9099/anat/sub-9099_ses-01_T1w.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9100/anat/sub-9100_ses-01_T1w.nii.gz'
]

zte_files = [
    root_path + '/transmission/data/ZTE/sub-9079/anat/sub-9079_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9085/anat/sub-9085_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9088/anat/sub-9088_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9089/anat/sub-9089_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9090/anat/sub-9090_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9091/anat/sub-9091_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9093/anat/sub-9093_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9094/anat/sub-9094_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9095/anat/sub-9095_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9096/anat/sub-9096_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9097/anat/sub-9097_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9098/anat/sub-9098_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9099/anat/sub-9099_ses-01_ZTE.nii.gz',
    root_path + '/transmission/data/ZTE/sub-9100/anat/sub-9100_ses-01_ZTE.nii.gz'
]

## Prepare ZTE Acoustic Simulation Inputs

This block scans the ZTE-tagged BabelBrain input files and creates the generic filenames expected by `RunAcousticSim`. It checks that the required ZTE mask, air-region mask, ZTE image, and calibration files exist before copying only the missing generic input files.

In [ ]:
from pathlib import Path
import pandas as pd
import nibabel as nib
import numpy as np

def qa_check_brainsight_txt(txt_path, expected_id):
    txt_path = Path(txt_path)

    if not txt_path.exists():
        raise FileNotFoundError(f"Missing Brainsight txt: {txt_path}")

    with open(txt_path, "r") as f:
        lines = f.readlines()

    nonempty = [x.strip() for x in lines if x.strip()]
    hits = [line for line in nonempty if str(expected_id) in line]

    print("\n[QA][Brainsight]")
    print(f"  file: {txt_path.name}")
    print(f"  expected ID: {expected_id}")
    print(f"  lines containing ID: {len(hits)}")

    for line in hits[:5]:
        print(f"    {line}")

    if len(hits) == 0:
        raise ValueError(f"{expected_id} not found in {txt_path.name}")

def qa_check_mask(mask_path, expected_id):
    mask_path = Path(mask_path)

    if not mask_path.exists():
        raise FileNotFoundError(f"Missing mask file: {mask_path}")

    img = nib.load(str(mask_path))
    data = img.get_fdata()

    target_vox = np.argwhere(data == 5.0)

    print("\n[QA][Mask]")
    print(f"  ID: {expected_id}")
    print(f"  mask: {mask_path.name}")
    print(f"  shape: {data.shape}")
    print(f"  target voxel count (label 5): {len(target_vox)}")

    if len(target_vox) == 0:
        raise ValueError(f"No label-5 target voxels found in {mask_path.name}")

    centroid_vox = target_vox.mean(axis=0)
    affine = img.affine
    centroid_mm = nib.affines.apply_affine(affine, centroid_vox)

    print(f"  centroid voxel: {np.round(centroid_vox, 2)}")
    print(f"  centroid world/mm: {np.round(centroid_mm, 2)}")

    return {
        "id": str(expected_id),
        "mask": str(mask_path),
        "n_target_vox": int(len(target_vox)),
        "centroid_vox": centroid_vox,
        "centroid_mm": centroid_mm
    }

# Main Loop

qa_results = []

for freq in frequencies:
    PPW = PPWs[freq]
    kHz = int(round(freq / 1e3))

    for T1W, eeg_csv, zte_file, sim_path in zip(t1w_files, eeg_files, zte_files, simbnibs_paths):
        T1W = Path(T1W)
        anat_dir = T1W.parent

        print("\n" + "="*60)
        print(f"[SUBJECT] T1W = {T1W.name}")
        print(f"[SUBJECT] EEG = {eeg_csv}")
        print(f"[SUBJECT] ZTE = {zte_file}")
        print(f"[SUBJECT] SIMBNIBS = {sim_path}")
        print("="*60)

        MatResults = ConvertEEGToBSightFUS(
            inputcsv=eeg_csv,
            outputBSight=str(anat_dir / "brainsight.txt"),
            depthFromSkin=depthFromSkin
        )

        # Read EEG CSV
        try:
            df = pd.read_csv(eeg_csv)
        except Exception:
            df = pd.read_csv(eeg_csv, sep=None, engine="python")

        id_col = None
        for c in [
            "Target Name", "target_name", "Target", "target",
            "Name", "name", "ID", "Id", "id",
            "Electrode", "electrode", "Label", "label"
        ]:
            if c in df.columns:
                id_col = c
                break

        if id_col is None:
            raise ValueError(
                f"Could not detect target ID column in EEG CSV: {eeg_csv}\n"
                f"Columns = {list(df.columns)}"
            )

        print(f"[INFO] Using ID column: {id_col}")
        print(f"[INFO] IDs from MatResults: {list(MatResults.keys())}")

        # Loop through each target separately
        for ID in MatResults:
            print("\n" + "-"*50)
            print(f"[TARGET] ID = {ID}")
            print("-"*50)
            
            row_df = df[df[id_col].astype(str) == str(ID)]

            if len(row_df) == 0:
                raise ValueError(f"Could not find ID={ID} in EEG CSV: {eeg_csv}")
            if len(row_df) > 1:
                raise ValueError(f"Multiple rows found for ID={ID} in EEG CSV: {eeg_csv}")

            # Save one-target csv
            single_csv = anat_dir / f"brainsight_{ID}.csv"
            row_df.to_csv(single_csv, index=False)
            print(f"[WRITE] single CSV: {single_csv.name}")

            # Convert one-target csv -> one-target brainsight txt
            single_bsight = anat_dir / f"brainsight_{ID}.txt"
            ConvertEEGToBSightFUS(
                inputcsv=str(single_csv),
                outputBSight=str(single_bsight),
                depthFromSkin=depthFromSkin
            )
            print(f"[WRITE] single Brainsight txt: {single_bsight.name}")

            qa_check_brainsight_txt(single_bsight, ID)

            # run mask generation
            RunMaskGeneration(
                T1W=str(T1W),
                ID=ID,
                simbnibs_path=str(sim_path),
                Mat4Trajectory=str(single_bsight),
                ComputingDevice=ComputingDevice,
                ComputingBackend=3,
                Frequency=freq,
                BasePPW=PPW,
                TxSystem=TxSystem,
                bUseCT=True,
                CT_or_ZTE_input=str(zte_file),
                ZTERange=[0.1, 0.65],
                HUThreshold=300.0,
                bReuseFiles=False,
                CTType=2
            )

            base = f"{ID}_{TxSystem}_{kHz}kHz_{PPW}PPW"
            mask_path = anat_dir / f"{base}_ZTE_BabelViscoInput.nii.gz"

            qa_result = qa_check_mask(mask_path, ID)
            qa_results.append(qa_result)

print("\n" + "="*70)
print("FINAL QA SUMMARY")
print("="*70)

for r in qa_results:
    print(
        f"ID={r['id']:<8} "
        f"n_target_vox={r['n_target_vox']:<8} "
        f"centroid_vox={np.round(r['centroid_vox'], 2)} "
        f"centroid_mm={np.round(r['centroid_mm'], 2)}"
    )

seen = {}
for r in qa_results:
    key = tuple(np.round(r["centroid_vox"], 2))
    seen.setdefault(key, []).append(r["id"])

print("\nPotential duplicate targets:")
found_duplicate = False
for key, ids in seen.items():
    if len(ids) > 1:
        found_duplicate = True
        print(f"  SAME centroid {key} -> IDs {ids}")

if not found_duplicate:
    print("  No duplicate centroids detected.")

print("\nZTE mask generation + QA done.")

## Run ZTE-based Acoustic Simulations

This block runs the ZTE-based acoustic simulations for each EEG-derived target and frequency. It converts each EEG trajectory into a Brainsight-style target, checks the ZTE acoustic input mask, verifies the target voxel label, and then runs the ZTE acoustic simulation workflow.

In [ ]:
from pathlib import Path
import shutil

for freq in frequencies:
    PPW = PPWs[freq]
    kHz = int(freq / 1e3)

    for T1W, eeg_csv in zip(t1w_files, eeg_files):
        T1W = Path(T1W)
        anat_dir = T1W.parent

        MatResults = ConvertEEGToBSightFUS(
            inputcsv=eeg_csv,
            outputBSight=str(anat_dir / "brainsight.txt"),
            depthFromSkin=depthFromSkin
        )

        for ID in MatResults:
            base = f"{ID}_{TxSystem}_{kHz}kHz_{PPW}PPW"

            # ZTE-tagged inputs produced by mask generation
            src_mask = anat_dir / f"{base}_ZTE_BabelViscoInput.nii.gz"
            src_air  = anat_dir / f"{base}_ZTE_AirRegions.nii.gz"
            src_ct   = anat_dir / f"{base}_ZTE.nii.gz"
            src_cal  = anat_dir / f"{base}_ZTE-cal.npz"

            for p in (src_mask, src_air, src_ct, src_cal):
                if not p.exists():
                    raise FileNotFoundError(f"Missing ZTE file: {p}")

            dst_mask = anat_dir / f"{base}_BabelViscoInput.nii.gz"
            dst_air  = anat_dir / f"{base}_AirRegions.nii.gz"
            dst_ct   = anat_dir / f"{base}_CT.nii.gz"
            dst_cal  = anat_dir / f"{base}_CT-cal.npz"

            # OVERWRITE to avoid stale generic files from older runs
            shutil.copyfile(src_mask, dst_mask)
            shutil.copyfile(src_air,  dst_air)
            shutil.copyfile(src_ct,   dst_ct)
            shutil.copyfile(src_cal,  dst_cal)

            print(f"[GENERIC][ZTE] {ID}")
            print(f"  {src_mask.name} -> {dst_mask.name}")
            print(f"  {src_air.name}  -> {dst_air.name}")
            print(f"  {src_ct.name}   -> {dst_ct.name}")
            print(f"  {src_cal.name}  -> {dst_cal.name}")

print("ZTE scan done (generic inputs refreshed).")

In [ ]:
from pathlib import Path
import nibabel as nib
import numpy as np

for freq in frequencies:
    PPW = PPWs[freq]
    kHz = int(round(freq / 1e3))
    foc_mm = float(FocalLength) * 1e3
    diam_mm = float(Aperture) * 1e3

    for T1W, eeg_csv in zip(t1w_files, eeg_files):
        T1W = Path(T1W)
        anat_dir = T1W.parent

        MatResults = ConvertEEGToBSightFUS(
            inputcsv=eeg_csv,
            outputBSight=str(anat_dir / "brainsight.txt"),
            depthFromSkin=depthFromSkin
        )

        for ID in MatResults:
            base = f"{ID}_{TxSystem}_{kHz}kHz_{PPW}PPW"

            # For ZTE, inspect the ZTE-tagged mask
            zte_mask = anat_dir / f"{base}_ZTE_BabelViscoInput.nii.gz"
            if not zte_mask.exists():
                raise FileNotFoundError(f"Missing acoustic input mask: {zte_mask}")

            img = nib.load(str(zte_mask))
            data = np.flip(img.get_fdata(), axis=2)

            pts = np.argwhere(data == 5.0)

            print("\n" + "=" * 90)
            print(f"[DEBUG] Subject : {T1W.name}")
            print(f"[DEBUG] ID      : {ID}")
            print(f"[DEBUG] Freq    : {freq}")
            print(f"[DEBUG] PPW     : {PPW}")
            print(f"[DEBUG] Mask    : {zte_mask.name}")
            print(f"[DEBUG] #voxels labeled 5 : {pts.shape[0]}")

            if pts.shape[0] == 0:
                raise RuntimeError(f"No target voxel (label 5) found in {zte_mask}")

            print(f"[DEBUG] first target voxel(s): {pts[:10]}")

            if pts.shape[0] > 1:
                print(f"[WARNING] Multiple target voxels found for {ID}. Acoustic code may be using the first one only.")

            target_preview = pts[0]
            line_of_sight = data[target_preview[0], target_preview[1], :]
            skin_idx = np.where(line_of_sight > 0)[0]

            if len(skin_idx) == 0:
                print(f"[WARNING] No skin found along line of sight for {ID}")
                start_skin = None
                distance_from_skin_mm = None
            else:
                start_skin = skin_idx.min()
                voxel_sizes = img.header.get_zooms()[:3]
                voxel_size_mm = float(voxel_sizes[0])
                distance_from_skin_mm = (target_preview[2] - start_skin) * voxel_size_mm

            print(f"[DEBUG] preview TargetLocation : {target_preview}")
            print(f"[DEBUG] preview StartSkin      : {start_skin}")
            print(f"[DEBUG] preview DistanceFromSkin (mm): {distance_from_skin_mm}")
            print("=" * 90)

            # Remove stale ZTE acoustic outputs before rerun
            old_patterns = [
                f"{base}_Foc{foc_mm:.1f}_Diam{diam_mm:.1f}_DataForSim*.h5",
                f"{base}_Foc{foc_mm:.1f}_Diam{diam_mm:.1f}_Water_DataForSim*.h5",
            ]
            for pat in old_patterns:
                for oldf in anat_dir.glob(pat):
                    try:
                        oldf.unlink()
                        print(f"[deleted old acoustic output] {oldf.name}")
                    except Exception as e:
                        print(f"[warning] could not delete {oldf.name}: {e}")

            RunAcousticSim(
                T1W=str(T1W),
                ID=ID,
                TxSystem=TxSystem,
                deviceName=ComputingDevice,
                Aperture=Aperture,
                FocalLength=FocalLength,
                Frequency=freq,
                PPW=PPW,
                COMPUTING_BACKEND=3,
                zLengthBeyonFocalPointWhenNarrow=zLengthBeyonFocalPointWhenNarrow,
                bUseCT=True,
                ModalityTag="ZTE"
            )

print("ZTE acoustic done.")

## Thermal Simulations

In [ ]:
from pathlib import Path
from multiprocessing import Process, Queue
import time
import yaml
import re
import h5py
import numpy as np

# Change to your Yaml path
THERM_YAML = "/Users/moonjeong/Desktop/Babelbrain/yaml/exp_1/Thermal_Profile_Calgary.yaml"

with open(THERM_YAML, "r") as f:
    ExtraData = yaml.safe_load(f)

BaseIsppa = 4.0  # W/cm2

AllDC_PRF_Duration = [{
    "DC": 0.1,
    "PRF": 100.0,
    "Duration": 90.0,
    "DurationOff": 210.0,
    "Repetitions": 2,
    "NumberGroupedSonications": 1,
    "PauseBetweenGroupedSonications": 0.0
}]

# Change to your own CT- and ZTE- rooth paths
CT_ROOT = Path("/Volumes/Trans_2/transmission/data/CT")
ZTE_ROOT = Path("/Volumes/Trans_2/transmission/data/ZTE")

def get_subject_id(path_like):
    """
    Extract sub-XXXX from a path string.
    """
    s = str(path_like)
    m = re.search(r"(sub-[A-Za-z0-9]+)", s)
    if not m:
        raise ValueError(f"Could not extract subject ID from path: {s}")
    return m.group(1)


def find_ratio_loss_anywhere(h5_path):
    """
    Recursively search the H5 for datasets whose name contains 'ratioloss'.
    Returns a dict of {dataset_path: value}.
    """
    out = {}

    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset) and "ratioloss" in name.lower():
            try:
                val = obj[()]
                if np.isscalar(val):
                    out[name] = float(val)
                else:
                    arr = np.asarray(val).squeeze()
                    if arr.size == 1:
                        out[name] = float(arr)
                    else:
                        out[name] = arr
            except Exception as e:
                out[name] = f"<unreadable: {e}>"

    with h5py.File(h5_path, "r") as f:
        f.visititems(visitor)

    return out


def run_thermal(h5_path, freq_hz):
    q = Queue()

    kargs = dict(
        COMPUTING_BACKEND=3,
        deviceName=ComputingDevice,
        TxSystem=TxSystem,
        Frequency=float(freq_hz),
        Isppa=float(BaseIsppa),
        sel_p="p_amp",
        BaselineTemperature=float(ExtraData.get("BaselineTemperature", 37.0)),
        LimitBHTEIterationsPerProcess=int(ExtraData.get("LimitBHTEIterationsPerProcess", 100)),
    )

    p = Process(
        target=CalculateThermalProcess,
        args=(q, str(h5_path), AllDC_PRF_Duration, ExtraData),
        kwargs=kargs
    )
    p.start()

    ok = True
    while p.is_alive():
        time.sleep(0.1)
        while not q.empty():
            msg = q.get()
            print(msg, end="")
            if "--Babel-Brain-Low-Error" in msg:
                ok = False

    p.join()

    while not q.empty():
        msg = q.get()
        print(msg, end="")
        if "--Babel-Brain-Low-Error" in msg:
            ok = False

    if not ok:
        raise RuntimeError(f"Thermal failed: {h5_path}")


for freq in frequencies:
    freq = float(freq)
    kHz = int(round(freq / 1e3))
    PPW = PPWs[freq]
    foc_mm = float(FocalLength) * 1e3
    diam_mm = float(Aperture) * 1e3

    print("\n" + "=" * 80)
    print(f"Frequency: {kHz} kHz | PPW: {PPW} | Foc: {foc_mm:.1f} mm | Diam: {diam_mm:.1f} mm")
    print("=" * 80)

    for T1W, eeg_csv in zip(t1w_files, eeg_files):
        subj = get_subject_id(T1W)
        subj_anat_dir = ZTE_ROOT / subj / "anat"

        print(f"\n--- Subject: {subj} ---")
        print(f"T1W: {T1W}")
        print(f"EEG CSV: {eeg_csv}")
        print(f"Searching only in: {subj_anat_dir}")

        if not subj_anat_dir.exists():
            print(f"[SKIP] anat folder not found: {subj_anat_dir}")
            continue

        MatResults = ConvertEEGToBSightFUS(
            inputcsv=eeg_csv,
            outputBSight=f"/tmp/{subj}_brainsight.txt",
            depthFromSkin=depthFromSkin
        )
        
        if isinstance(MatResults, dict):
            IDs = list(MatResults.keys())
        else:
            IDs = list(MatResults)

        print(f"Found {len(IDs)} EEG targets")

        for ID in IDs:
            pattern = (
                f"{ID}_{TxSystem}_{kHz}kHz_{PPW}PPW_"
                f"Foc{foc_mm:.1f}_Diam{diam_mm:.1f}_DataForSim*.h5"
            )

            hits = sorted(subj_anat_dir.glob(pattern))

            print(f"\nID: {ID}")
            print(f"Pattern: {pattern}")

            if len(hits) == 0:
                print(f"[MISSING] No H5 found for {subj} | {ID}")
                continue

            if len(hits) > 1:
                print("[AMBIGUOUS] More than one H5 matched:")
                for h in hits:
                    print("   ", h.name)
                raise RuntimeError(
                    f"Multiple H5 files matched for {subj}, ID={ID}. "
                    f"Make the filename pattern more specific."
                )

            h5 = hits[0]
            print(f"Using H5: {h5}")

            # Inspect stored RatioLoss before thermal
            try:
                rl = find_ratio_loss_anywhere(h5)
                if rl:
                    print("Stored RatioLoss dataset(s):")
                    for k, v in rl.items():
                        print(f"   {k}: {v}")
                else:
                    print("No dataset containing 'RatioLoss' found in this H5.")
            except Exception as e:
                print(f"Could not inspect RatioLoss in {h5.name}: {e}")

            # Run thermal on this exact subject-specific H5
            run_thermal(h5, freq)

# Remove generic CT and ZTE files

In [ ]:
from pathlib import Path

# Root folder that contains sub-XXXX/anat (edit if needed)
ROOT = Path("/Volumes/EXTERNAL/trasmission/data/CT")
ROOT = Path("/Volumes/EXTERNAL/trasmission/data/ZTE")

for anat_dir in ROOT.glob("sub-*/anat"):
    # generic files you might have created
    for gen in anat_dir.glob("*_BabelViscoInput.nii.gz"):
        # skip already-tagged originals
        if "_CT_" in gen.name or "_ZTE_" in gen.name:
            continue

        ct_src = gen.with_name(gen.name.replace("_BabelViscoInput.nii.gz", "_CT_BabelViscoInput.nii.gz"))
        if ct_src.exists():
            gen.unlink()  # delete generic
            # print("removed", gen)

    for gen in anat_dir.glob("*_AirRegions.nii.gz"):
        if "_CT_" in gen.name or "_ZTE_" in gen.name:
            continue

        ct_src = gen.with_name(gen.name.replace("_AirRegions.nii.gz", "_CT_AirRegions.nii.gz"))
        if ct_src.exists():
            gen.unlink()
            print("removed", gen)

print("Cleanup done (removed only generic BabelViscoInput/AirRegions when CT originals exist).")